In [ ]:
#Insurance Inference: Algunos tópicos de inferencia con python y nuestra base de datos insurance
#python -m pip install ipykernel

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from scipy import stats
import plotly.graph_objects as go
import random
import altair as alt
alt.data_transformers.enable("vegafusion")
from scipy.stats import t
from statistics import variance
rng = np.random.default_rng()
x = stats.uniform.rvs(size=75, random_state=rng)
sns.set(style="darkgrid")

In [2]:
df = pd.read_csv('insurance.csv')

In [3]:
df.shape

(1338, 7)

In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1338 entries, 0 to 1337
Data columns (total 7 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   age       1338 non-null   int64  
 1   sex       1338 non-null   object 
 2   bmi       1338 non-null   float64
 3   children  1338 non-null   int64  
 4   smoker    1338 non-null   object 
 5   region    1338 non-null   object 
 6   charges   1338 non-null   float64
dtypes: float64(2), int64(2), object(3)
memory usage: 73.3+ KB


In [5]:
df.head()

,age,sex,bmi,children,smoker,region,charges
0,19,female,27.900,0,yes,southwest,16884.92400
1,18,male,33.770,1,no,southeast,1725.55230
2,28,male,33.000,3,no,southeast,4449.46200
3,33,male,22.705,0,no,northwest,21984.47061
4,32,male,28.880,0,no,northwest,3866.85520


In [6]:
#Muestreo. Distribuciones de muestreo comunes en estadística
#Muestra de una variable continua

np.random.seed(123)

df.sample(n=20)["bmi"]


650     42.680
319     37.335
314     31.400
150     24.130
336     25.740
970     28.160
169     18.905
684     18.500
1097    33.770
512     22.420
597     33.250
698     33.725
630     36.100
738     31.730
696     32.300
827     28.025
18      40.300
31      26.315
682     35.300
185     41.895
Name: bmi, dtype: float64

In [7]:
np.random.seed(123)

df.sample(n=20)["bmi"].describe()

count    20.00000
mean     31.09900
std       7.02806
min      18.50000
25%      26.17125
50%      32.01500
75%      35.50000
max      42.68000
Name: bmi, dtype: float64

In [8]:
#De una variable discreta
np.random.seed(123)
df.sample(n=40)["sex"].value_counts()

sex
male      24
female    16
Name: count, dtype: int64

In [9]:
#Proporciones
np.random.seed(123)
df.sample(n=40)["sex"].value_counts(normalize=True)

sex
male      0.6
female    0.4
Name: proportion, dtype: float64

In [10]:
np.random.seed(123)
df.sample(n=100)["region"].value_counts()

region
southeast    28
northeast    28
southwest    28
northwest    16
Name: count, dtype: int64

In [11]:
df["region"].value_counts()

region
southeast    364
southwest    325
northwest    325
northeast    324
Name: count, dtype: int64

In [12]:
np.random.seed(123)
df.sample(n=100)["region"].value_counts(normalize=True)

region
southeast    0.28
northeast    0.28
southwest    0.28
northwest    0.16
Name: proportion, dtype: float64

In [13]:
#Una muestra de tamaño 20 de toda la base
np.random.seed(123)
samples = pd.concat([
    df.sample(20).assign(replicate=n)
    for n in range(1)
])
samples

,age,sex,bmi,children,smoker,region,charges,replicate
650,49,female,42.680,2,no,southeast,9800.88820,0
319,32,male,37.335,1,no,northeast,4667.60765,0
314,27,female,31.400,0,yes,southwest,34838.87300,0
150,35,male,24.130,1,no,northwest,5125.21570,0
336,60,male,25.740,0,no,southeast,12142.57860,0
970,50,female,28.160,3,no,southeast,10702.64240,0
169,27,male,18.905,3,no,northeast,4827.90495,0
684,33,female,18.500,1,no,southwest,4766.02200,0
1097,22,male,33.770,0,no,southeast,1674.63230,0
512,51,male,22.420,0,no,northeast,9361.32680,0


In [14]:
#Muchas muestras: 500 muestras de tamaño 20
np.random.seed(123)
samples500 = pd.concat([
    df.sample(20).assign(replicate=n)
    for n in range(500)
])
samples500


,age,sex,bmi,children,smoker,region,charges,replicate
650,49,female,42.680,2,no,southeast,9800.88820,0
319,32,male,37.335,1,no,northeast,4667.60765,0
314,27,female,31.400,0,yes,southwest,34838.87300,0
150,35,male,24.130,1,no,northwest,5125.21570,0
336,60,male,25.740,0,no,southeast,12142.57860,0
...,...,...,...,...,...,...,...,...
1077,21,male,26.030,0,no,northeast,2102.26470,499
1236,63,female,21.660,0,no,northeast,14449.85440,499
182,22,male,19.950,3,no,northeast,4005.42250,499
19,30,male,35.300,0,yes,southwest,36837.46700,499


In [15]:
#Distribución de muestreo sobre la proporción de sexo (sex)
(
    samples500
    .groupby("replicate")
    ["sex"]
    .value_counts(normalize=True) 
)


replicate  sex   
0          male      0.65
           female    0.35
1          female    0.65
           male      0.35
2          female    0.55
                     ... 
497        male      0.45
498        female    0.55
           male      0.45
499        male      0.60
           female    0.40
Name: proportion, Length: 1000, dtype: float64

In [16]:
(
    samples500
    .groupby("replicate")
    ["sex"]
    .value_counts(normalize=True)
    .reset_index(name="Proporción muestral")
)

,replicate,sex,Proporción muestral
0,0,male,0.65
1,0,female,0.35
2,1,female,0.65
3,1,male,0.35
4,2,female,0.55
...,...,...,...
995,497,male,0.45
996,498,female,0.55
997,498,male,0.45
998,499,male,0.60


In [17]:
sample_est = (
    samples500
    .groupby("replicate")
    ["sex"]
    .value_counts(normalize=True)
    .reset_index(name="Proporción muestral")
)
sample_est = sample_est[sample_est["sex"] == "female"]
sample_est

,replicate,sex,Proporción muestral
1,0,female,0.35
2,1,female,0.65
4,2,female,0.55
7,3,female,0.45
9,4,female,0.40
...,...,...,...
991,495,female,0.45
992,496,female,0.60
994,497,female,0.55
996,498,female,0.55


In [18]:
#pip install "vl-convert-python>=1.9.0"

In [19]:
sampling_dist = alt.Chart(sample_est).mark_bar().encode(
    x=alt.X("Proporción muestral")
        .bin(maxbins=30)
        .title("Sexo femenino: Proporciones muestrales"),
    y=alt.Y("count()").title("Count"),
)

sampling_dist

alt.Chart(...)

In [20]:
#El estimador puntual de esta proporción estas muestras
sample_est["Proporción muestral"].mean()

np.float64(0.5005)

In [21]:
df.sex.value_counts(normalize=True)

sex
male      0.505232
female    0.494768
Name: proportion, dtype: float64

In [22]:
#Intervalo de confianza
data=pd.DataFrame(sample_est["Proporción muestral"])
confidence = 0.95
df = len(data) -1 #grados de libeta'
mean = np.mean(data)
std_err = np.std(data, ddof=1) / np.sqrt(len(data)) 

# Intervalo de confianza
ci = t.interval(confidence, df, loc=mean, scale=std_err)
print("95% Confidence Interval:", ci)


95% Confidence Interval: (array([0.49083923]), array([0.51016077]))


/Users/valeria/anaconda3/envs/diplomado/lib/python3.11/site-packages/numpy/_core/fromnumeric.py:3800: FutureWarning: The behavior of DataFrame.std with axis=None is deprecated, in a future version this will reduce over both axes and return a scalar. To retain the old behavior, pass axis=0 (or do not pass axis)
  return std(axis=axis, dtype=dtype, out=out, ddof=ddof, **kwargs)


In [23]:
#Distribución muestral para variables continuas
#Distribución de la variable original
#Algo pasó y tuve que volver a cargar la base
df = pd.read_csv('insurance.csv')

In [24]:
charges_distribution = alt.Chart(df).mark_bar().encode(
    x=alt.X("charges")
        .bin(maxbins=40)
        .title("Charges"),
    y=alt.Y("count()", title="Count"),
)

charges_distribution

alt.Chart(...)

In [24]:
df["charges"].mean()

np.float64(13270.422265141257)

In [26]:
#Una muestra de toda la base
np.random.seed(123)
df_one_sample = df.sample(n=100)

In [27]:
sample_distribution = alt.Chart(df_one_sample).mark_bar().encode(
    x=alt.X("charges")
        .bin(maxbins=40)
        .title("Charges"),
    y=alt.Y("count()").title("Count"),
)

sample_distribution

alt.Chart(...)

In [28]:
df_one_sample["charges"].mean()

np.float64(13905.618141590003)

In [29]:
#Distribución muestral charges
charges_estimates = (
    samples500
    .groupby("replicate")
    ["charges"]
    .mean()
    .reset_index()
    .rename(columns={"charges": "mean_charges"})
)
charges_estimates

,replicate,mean_charges
0,0,15368.582128
1,1,8843.910949
2,2,10100.343948
3,3,11973.103941
4,4,14215.958155
...,...,...
495,495,10817.586955
496,496,13795.989945
497,497,13827.848310
498,498,13642.121874


In [30]:
charges_distribution = alt.Chart(charges_estimates).mark_bar().encode(
    x=alt.X("mean_charges")
        .bin(maxbins=30)
        .title("Media muestral charges"),
    y=alt.Y("count()").title("Count")
)

charges_distribution

alt.Chart(...)

In [31]:
charges_estimates["mean_charges"].mean()

np.float64(13217.7579901312)

In [32]:
#Intervalo de confianza
data1=pd.DataFrame(charges_estimates["mean_charges"])
confidence = 0.95
df = len(data1) -1 
mean = np.mean(data1)
std_err = np.std(data1, ddof=1) / np.sqrt(len(data1)) 

# Intervalo de confianza
ci = t.interval(confidence, df, loc=mean, scale=std_err)
print("95% Confidence Interval:", ci)

95% Confidence Interval: (array([12994.78478068]), array([13440.73119958]))


/Users/valeria/anaconda3/envs/diplomado/lib/python3.11/site-packages/numpy/_core/fromnumeric.py:3800: FutureWarning: The behavior of DataFrame.std with axis=None is deprecated, in a future version this will reduce over both axes and return a scalar. To retain the old behavior, pass axis=0 (or do not pass axis)
  return std(axis=axis, dtype=dtype, out=out, ddof=ddof, **kwargs)


In [33]:
#Distribuciones de muestreo: Varianza muestral 
charges_estimates1 = (
    samples500
    .groupby("replicate")
    ["charges"]
    .var()
    .reset_index()
    .rename(columns={"charges": "var_charges"})
)
charges_estimates1

,replicate,var_charges
0,0,1.852059e+08
1,1,4.864022e+07
2,2,1.207756e+08
3,3,8.919260e+07
4,4,2.294739e+08
...,...,...
495,495,1.159879e+08
496,496,1.374398e+08
497,497,1.127169e+08
498,498,2.069158e+08


In [34]:
charges_var_distribution = alt.Chart(charges_estimates1).mark_bar().encode(
    x=alt.X("var_charges")
        .bin(maxbins=30)
        .title("Varianza muestral charges"),
    y=alt.Y("count()").title("Count")
)

charges_var_distribution

alt.Chart(...)

In [35]:
#
charges_estimates1["var_charges"].mean()

np.float64(148848377.71734372)

In [36]:
df = pd.read_csv('insurance.csv')

In [37]:
np.var(df["charges"])

146542766.49354774

In [38]:
 #Intervalo de confianza
data1=pd.DataFrame(charges_estimates1["var_charges"])
alpha = 0.05
n = len(data1)
var = np.mean(data1)
chi2_lower = stats.chi2.ppf(alpha / 2, df=n - 1)
chi2_upper = stats.chi2.ppf(1 - alpha / 2, df=n - 1) 
ci_lower = (n - 1)*var/chi2_upper
ci_upper = (n - 1)*var/chi2_lower
print(f"95% Intervalo de confianza de la varianza: ({ci_lower:.4f}, {ci_upper:.4f})")

95% Intervalo de confianza de la varianza: (131977126.4562, 169192881.7151)


In [39]:
#Bootstrap. Asumamos que nuestros datos son los que se encuentran en df_one_sample

df_one_sample


,age,sex,bmi,children,smoker,region,charges
650,49,female,42.680,2,no,southeast,9800.888200
319,32,male,37.335,1,no,northeast,4667.607650
314,27,female,31.400,0,yes,southwest,34838.873000
150,35,male,24.130,1,no,northwest,5125.215700
336,60,male,25.740,0,no,southeast,12142.578600
...,...,...,...,...,...,...,...
427,18,female,29.165,0,no,northeast,7323.734819
163,32,female,29.800,2,no,southwest,5152.134000
196,39,female,32.800,0,no,southwest,5649.715000
863,36,female,19.855,0,no,northeast,5458.046450


In [40]:
#pip install "vegafusion>=2.0.3"

In [41]:
one_sample_dist = alt.Chart(df_one_sample).mark_bar().encode(
    x=alt.X("charges")
        .bin(maxbins=30)
        .title("Charges"),
    y=alt.Y("count()").title("Count"),
)

one_sample_dist

alt.Chart(...)

In [42]:
#Bootstrap con esta variable charges. Una sola remuestra

np.random.seed(123)
boot_charges = df_one_sample.sample(frac=1, replace=True)
boot_charges_dist = alt.Chart(boot_charges).mark_bar().encode(
    x=alt.X("charges")
        .bin(maxbins=30)
        .title("Boot charges"),
    y=alt.Y("count()", title="Count"),
)

boot_charges

,age,sex,bmi,children,smoker,region,charges
547,54,female,46.700,2,no,southwest,11538.42100
167,32,female,33.155,3,no,northwest,6128.79745
863,36,female,19.855,0,no,northeast,5458.04645
31,18,female,26.315,0,no,northeast,2198.18985
13,56,female,39.820,0,no,southeast,11090.71780
...,...,...,...,...,...,...,...
1027,23,male,18.715,0,no,northwest,21595.38229
169,27,male,18.905,3,no,northeast,4827.90495
512,51,male,22.420,0,no,northeast,9361.32680
1094,50,female,33.700,4,no,southwest,11299.34300


In [43]:
boot_charges_dist

alt.Chart(...)

In [44]:
boot_charges["charges"].mean()

np.float64(14027.199331800002)

In [45]:
#Muchas remuestras
boot_charges_10000 = pd.concat([
    df_one_sample.sample(frac=1, replace=True).assign(replicate=n)
    for n in range(10_000)
])
boot_charges_10000

,age,sex,bmi,children,smoker,region,charges,replicate
13,56,female,39.820,0,no,southeast,11090.71780,0
853,53,female,23.750,2,no,northeast,11729.67950,0
630,53,male,36.100,1,no,southwest,10085.84600,0
921,62,female,33.200,0,no,southwest,13462.52000,0
1104,37,male,29.800,0,no,southwest,20420.60465,0
...,...,...,...,...,...,...,...,...
1256,51,female,36.385,3,no,northwest,11436.73815,9999
1027,23,male,18.715,0,no,northwest,21595.38229,9999
1326,42,female,32.870,0,no,northeast,7050.02130,9999
1120,23,female,31.400,0,yes,southwest,34166.27300,9999


In [46]:
#Algunos histogramas de estas muestras bootstrap
four_bootstrap_samples = boot_charges_10000.query("replicate < 4")
four_bootstrap_fig = alt.Chart(four_bootstrap_samples, height=150).mark_bar().encode(
    x=alt.X("charges")
        .bin(maxbins=20)
        .title("Charges"),
    y=alt.Y("count()").title("Count")
).facet(
    "replicate:N",
    columns=2
)
four_bootstrap_fig


alt.FacetChart(...)

In [47]:
#Media de cada una de estas primeras 4 remuestra (replicate)
(
    four_bootstrap_samples
    .groupby("replicate")
    ["charges"]
    .mean()
    .reset_index()
    .rename(columns={"charges": "mean_charges"})
)

,replicate,mean_charges
0,0,12003.343498
1,1,12828.923887
2,2,14514.303901
3,3,14436.014825


In [48]:
#Media de todas las 10000 remuestras (replicate)
boot_charges_10000_means = (
    boot_charges_10000
    .groupby("replicate")
    ["charges"]
    .mean()
    .reset_index()
    .rename(columns={"charges": "mean_charges"})
)

boot_charges_10000_means

,replicate,mean_charges
0,0,12003.343498
1,1,12828.923887
2,2,14514.303901
3,3,14436.014825
4,4,14353.072886
...,...,...
9995,9995,14996.609394
9996,9996,13658.869680
9997,9997,14035.812187
9998,9998,14693.899624


In [49]:
#pip install "vl-convert-python>=1.6.0"

In [50]:
boot_charges_dist = alt.Chart(boot_charges_10000_means).mark_bar().encode(
    x=alt.X("mean_charges")
        .bin(maxbins=20)
        .title("Bootstrap media charges"),
    y=alt.Y("count()").title("Count"),
)

boot_charges_dist

alt.Chart(...)

In [51]:
boot_charges_10000_means["mean_charges"].mean()

np.float64(13918.188857536503)

In [52]:
#Intervalo de confianza empírico (Este intervalo es un intervalo los percentiles de la distribución empírica)
ci_boot_charges = boot_charges_10000_means["mean_charges"].quantile([0.025, 0.975])
ci_boot_charges

0.025    11652.486222
0.975    16389.988614
Name: mean_charges, dtype: float64

In [ ]:
#VaR (value at risk) y es el cuantil 95 de los datos

#VaR de todas las 10000 remuestras (replicate)
boot_charges_10000_VaR = (
    boot_charges_10000
    .groupby("replicate")
    ["charges"]
    .quantile(0.95)
    .reset_index()
    .rename(columns={"charges": "VaR_charges"})
)

boot_charges_10000_VaR


,replicate,VaR_charges
0,0,36151.464410
1,1,45702.022350
2,2,43850.771315
3,3,40286.362352
4,4,36291.926278
...,...,...
9995,9995,40286.362352
9996,9996,35116.956354
9997,9997,36149.483500
9998,9998,38618.414724


In [54]:
boot_charges_10000_VaR["VaR_charges"].mean()

np.float64(38921.22856496185)

In [55]:
#Intervalo de confianza empírico VaR(Este intervalo es un intervalo los percentiles de la distribución empírica)
ci_boot_charges_VaR = boot_charges_10000_VaR["VaR_charges"].quantile([0.025, 0.975])
ci_boot_charges_VaR

0.025    34265.433525
0.975    45813.322732
Name: VaR_charges, dtype: float64

In [56]:
df = pd.read_csv('insurance.csv')
np.percentile(df["charges"],95)

np.float64(41181.827787499926)